# Hybrid RCSP-Rie Architecture

In [ ]:
from sklearnex import patch_sklearn

patch_sklearn()

import numpy as np
import scipy.io
import os

from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.covariance import LedoitWolf

from pyriemann.estimation import Covariances
from pyriemann.classification import TSclassifier
from pyriemann.spatialfilters import CSP
from pyriemann.tangentspace import TangentSpace
#from pyriemann.classification import SVC

from sklearn.svm import SVC

import h5py

import time

In [ ]:
start_time = time.time()

In [ ]:
test_sub_ids = list(range(18))  

In [ ]:
Nc = 108

# Load datasets

In [ ]:
import os
import h5py
import numpy as np
import scipy.io

def load_subject_data(loading_path, subject_id, n_components=108):
    
    training_dataset_path = os.path.join(loading_path, 'TrainingDataset.mat')
    testing_dataset_path = os.path.join(loading_path, 'TestingDataset.mat') 
    training_projected_path = os.path.join(loading_path, 'TrainingProjected.mat')
    testing_projected_path = os.path.join(loading_path, 'TestingProjected.mat')
    training_features_path = os.path.join(loading_path, 'TrainingFeatures.mat')
    testing_features_path = os.path.join(loading_path, 'TestingFeatures.mat')
   
    # Load Training Dataset
    with h5py.File(training_dataset_path, 'r') as f_training_dataset:
        training_dataset = f_training_dataset['TrainingDataset']
        subject_cell_ref = training_dataset[subject_id, 0]
        subject_data = f_training_dataset[subject_cell_ref]
        X_train_dataset = np.transpose(subject_data['x'][:], (0, 1, 2))
        Y_train_dataset = np.squeeze(subject_data['y'][:])

    # Load Testing Dataset
    with h5py.File(testing_dataset_path, 'r') as f_testing_dataset:
        testing_dataset = f_testing_dataset['TestingDataset']
        subject_cell_ref = testing_dataset[subject_id, 0]
        subject_data = f_testing_dataset[subject_cell_ref]
        X_test_dataset = np.transpose(subject_data['x'][:], (0, 1, 2))
        Y_test_dataset = np.squeeze(subject_data['y'][:])

    # Load Training Projected Dataset
    with h5py.File(training_projected_path, 'r') as f_training_projected:
        training_projected = f_training_projected['TrainingProjected']
        subject_cell_ref = training_projected[subject_id, 0]
        subject_data = f_training_projected[subject_cell_ref]
        X_train_projected = np.transpose(subject_data['x'][:], (0, 2, 1))
        Y_train_projected = np.squeeze(subject_data['y'][:])

    # Load Testing Projected Dataset
    with h5py.File(testing_projected_path, 'r') as f_testing_projected:
        testing_projected = f_testing_projected['TestingProjected']
        subject_cell_ref = testing_projected[subject_id, 0]
        subject_data = f_testing_projected[subject_cell_ref]
        X_test_projected = np.transpose(subject_data['x'][:], (0, 2, 1))
        Y_test_projected = np.squeeze(subject_data['y'][:])
            
    # Load Training Features
    f_training_features = scipy.io.loadmat(training_features_path)
    training_features = f_training_features['TrainingFeatures'][0, subject_id]
    X_train_features = training_features[:, :n_components]
    Y_train_features = training_features[:, n_components:].flatten()

    # Load Testing Features
    f_testing_features = scipy.io.loadmat(testing_features_path)
    testing_features = f_testing_features['TestingFeatures'][0, subject_id]
    X_test_features = testing_features[:, :n_components]
    Y_test_features = testing_features[:, n_components:].flatten()

    # Organize results into a dictionary
    data = {
        "X_train_dataset": X_train_dataset,
        "Y_train_dataset": Y_train_dataset,
        "X_test_dataset": X_test_dataset,
        "Y_test_dataset": Y_test_dataset,
        "X_train_projected": X_train_projected,
        "Y_train_projected": Y_train_projected,
        "X_test_projected": X_test_projected,
        "Y_test_projected": Y_test_projected,
        "X_train_features": X_train_features,
        "Y_train_features": Y_train_features,
        "X_test_features": X_test_features,
        "Y_test_features": Y_test_features,
    }

    return data


# Classification Algorithms 

## Seq. RCSP-Rie
### Project CSP spatially filtered signals to tangent space 

In [ ]:
def SeqRcs(X_train_projected, Y_train_projected, X_test_projected, Y_test_dataset, metric='riemann', C=1.0, kernel='linear'):
    # Estimate covariance matrices
    cov_estimator_projected = Covariances(estimator='lwf')  
    X_train_cov_projected = cov_estimator_projected.transform(X_train_projected)
    X_test_cov_projected = cov_estimator_projected.transform(X_test_projected)
    

    # Train the classifier
    clf_TS_proj = TSclassifier(metric=metric, clf=SVC(C=C, kernel=kernel, class_weight='balanced'))
    clf_TS_proj.fit(X_train_cov_projected, Y_train_projected)

    # Make predictions on the test set
    y_pred_proj = clf_TS_proj.predict(X_test_cov_projected)
    
    # Compute performance metrics
    accuracy = accuracy_score(Y_test_dataset, y_pred_proj) 
    recall = recall_score(Y_test_dataset, y_pred_proj) 
    precision = precision_score(Y_test_dataset, y_pred_proj) 
    f1 = f1_score(Y_test_dataset, y_pred_proj) 
    
    # Print metrics
    print(f"Seq RCSP-Rie A: {accuracy:.4f}%")
    print(f"Seq RCSP-Rie R: {recall:.4f}%")
    print(f"Seq RCSP-Rie P: {precision:.4f}%")
    print(f"Seq RCSP-Rie F1: {f1:.4f}%")

    # Return metrics as a dictionary
    metrics = {
        "accuracy": accuracy,
        "recall": recall,
        "precision": precision,
        "f1_score": f1,
    }
    return metrics

## Par. RCSP-Rie
### Concatenate RCSP spatially filtered features and tangent space features 

In [ ]:
def ParRCSPRie(X_train_dataset, Y_train_dataset, X_test_dataset, Y_test_dataset, 
                                    X_train_features, X_test_features, metric='riemann', C=1.0, kernel='linear'):

    # --- Find tanget space features ---
    # Covariance matrix estimation
    cov_estimator = Covariances(estimator='lwf')  
    X_train_cov = cov_estimator.transform(X_train_dataset)
    X_test_cov = cov_estimator.transform(X_test_dataset)

    # Tangent Space transformation
    TSpace = TangentSpace(metric=metric, tsupdate=False).fit(X_train_cov)
    X_train_TS = TSpace.transform(X_train_cov)
    X_test_TS = TSpace.transform(X_test_cov)
    
    # --- Concatenate Spat filt features and TS features ---
    X_train_cat = np.concatenate((X_train_features, X_train_TS), axis=1)
    X_test_cat = np.concatenate((X_test_features, X_test_TS), axis=1)

    # Train the classifier
    clf_TS_cat = SVC(C=C, kernel=kernel, class_weight='balanced')
    clf_TS_cat.fit(X_train_cat, Y_train_dataset)

    # Make predictions on the test set
    y_pred_cat = clf_TS_cat.predict(X_test_cat)
    
    # Compute performance metrics
    accuracy = accuracy_score(Y_test_dataset, y_pred_cat) 
    recall = recall_score(Y_test_dataset,y_pred_cat) 
    precision = precision_score(Y_test_dataset, y_pred_cat) 
    f1 = f1_score(Y_test_dataset, y_pred_cat) 

    # Print metrics
    print(f"Par RCSP-Rie A: {accuracy:.4f}%")
    print(f"Par RCSP-Rie R: {recall:.4f}%")
    print(f"Par RCSP-Rie P: {precision:.4f}%")
    print(f"Par RCSP-Rie F1: {f1:.4f}%")

    
    # Return metrics as a dictionary
    metrics = {
        "accuracy": accuracy,
        "recall": recall,
        "precision": precision,
        "f1_score": f1,
    }
    
    return metrics

# Implement functions

In [ ]:
Results = {}

loading_paths = [
    ("LoadingPath_Non", "No"), 
    ("LoadingPath_PS", "Yes")  
]



for loading_path, pre_alignment_condition in loading_paths:
    print(f"Processing {loading_path} with PreAlignmentCondition = {pre_alignment_condition}")
    
    # Loop through each subject
    for subject_id in test_sub_ids:
        print(f"Loading data for Subject {subject_id}...")
        
        # Load the subject data
        subject_data = load_subject_data(loading_path, subject_id, n_components=Nc, PreAlignment=pre_alignment_condition)
        
        # Dataset
        X_train_dataset = subject_data["X_train_dataset"]
        Y_train_dataset = subject_data["Y_train_dataset"]
        X_test_dataset = subject_data["X_test_dataset"]
        Y_test_dataset = subject_data["Y_test_dataset"]
        
        # Projected
        X_train_projected = subject_data["X_train_projected"]
        Y_train_projected = subject_data["Y_train_projected"]
        X_test_projected = subject_data["X_test_projected"]
        Y_test_projected = subject_data["Y_test_projected"]
        
        # Features
        X_train_features = subject_data["X_train_features"]
        Y_train_features = subject_data["Y_train_features"]
        X_test_features = subject_data["X_test_features"]
        Y_test_features = subject_data["Y_test_features"]
        
        # Sanity check
        print(X_train_dataset.shape)
        print(Y_train_dataset.shape)
        print(X_test_dataset.shape)
        print(Y_test_dataset.shape)

        print(X_train_projected.shape) 
        print(Y_train_projected.shape)
        print(X_test_projected.shape) 
        print(Y_test_projected.shape)

        print(X_train_features.shape) 
        print(Y_train_features.shape)
        print(X_test_features.shape)
        print(Y_test_features.shape)
        
        # Implement classification methods

        SeqRCSPRie_Metrics = SeqRCSPRie(X_train_projected, Y_train_projected, X_test_projected, Y_test_dataset, metric='riemann', C=1.0, kernel='linear')
        ParRCSPRie_Metrics = ParRCSPRie(X_train_dataset, Y_train_dataset, X_test_dataset, Y_test_dataset, X_train_features, X_test_features, metric='riemann', C=1.0, kernel='linear')
        
        # Store the results for each subject
        if subject_id not in Results:
            Results[subject_id] = {}
        
        Results[subject_id][f"{loading_path}_SeqRCSPRie_Metrics"] = SeqRCSPRie_Metrics
        Results[subject_id][f"{loading_path}_ParRCSPRie_Metrics"] = ParRCSPRie_Metrics
        
        # Clear variables to free memory
        del (
            X_train_dataset, Y_train_dataset,
            X_test_dataset, Y_test_dataset,
            X_train_projected, Y_train_projected,
            X_test_projected, Y_test_projected,
            X_train_features, Y_train_features,
            X_test_features, Y_test_features
        )
        
        print(f"Cleared data for Subject {subject_id}.")
    
print(Results)

In [ ]:
end_time = time.time()
elapsed_time = end_time - start_time
print("Elapsed time:", elapsed_time/60, "minutes")